
# Bidirectional LSTM from Scratch (Raw PyTorch Tensor Ops)

**Objective:** Understand sequence transduction and temporal memory tracking by
manually implementing the four LSTM gate equations (Input, Forget, Output,
Candidate Cell State) and combining a forward and a backward recurrence into a
bidirectional layer, using fixed-length sequences.

**Constraints:** No `nn.RNN` / `nn.LSTM` is used for the core recurrence logic.
Only raw tensor operations (`torch.matmul`, `torch.sigmoid`, `torch.tanh`,
indexing, concatenation) are used to build the recurrence. `nn.LSTM` is used
**only** at the end, as a reference implementation to validate correctness.

## Contents
1. LSTM gate equations (theory)
2. Manual single-direction LSTM recurrence
3. Manual bidirectional LSTM layer (forward + backward + concat)
4. Validation against `nn.LSTM(bidirectional=True)` (shapes + values)
5. Sanity checks (gradient flow, different seq lengths/batch sizes)



## 1. LSTM Gate Equations

For a single direction, at timestep $t$, given input $x_t$ and previous
hidden/cell state $h_{t-1}, c_{t-1}$:

$$
\begin{aligned}
i_t &= \sigma(W_{ii} x_t + b_{ii} + W_{hi} h_{t-1} + b_{hi}) && \text{(Input gate)}\\
f_t &= \sigma(W_{if} x_t + b_{if} + W_{hf} h_{t-1} + b_{hf}) && \text{(Forget gate)}\\
g_t &= \tanh(W_{ig} x_t + b_{ig} + W_{hg} h_{t-1} + b_{hg}) && \text{(Candidate cell state)}\\
o_t &= \sigma(W_{io} x_t + b_{io} + W_{ho} h_{t-1} + b_{ho}) && \text{(Output gate)}\\
c_t &= f_t \odot c_{t-1} + i_t \odot g_t && \text{(New cell state)}\\
h_t &= o_t \odot \tanh(c_t) && \text{(New hidden state)}
\end{aligned}
$$

We stack the four gates' weights into single matrices
$W_{ih} \in \mathbb{R}^{4H \times I}$ and $W_{hh} \in \mathbb{R}^{4H \times H}$
(order **i, f, g, o**) to match PyTorch's internal `nn.LSTM` weight layout,
which lets us copy weights directly for a fair, exact comparison later.


In [1]:

import torch
import torch.nn as nn

torch.manual_seed(0)
device = torch.device("cpu")
print(torch.__version__)


2.13.0+cu130



## 2. Manual single-direction LSTM recurrence

`manual_lstm_direction` runs one direction (forward as given) of the
recurrence over a sequence using only tensor ops. It expects weights already
in PyTorch's stacked `[i, f, g, o]` order so we can validate against
`nn.LSTM` later.

Shapes:
- `x`: `(batch, seq_len, input_size)`
- `w_ih`: `(4*hidden, input_size)`, `w_hh`: `(4*hidden, hidden)`
- `b_ih`, `b_hh`: `(4*hidden,)`


In [2]:

def manual_lstm_direction(x, w_ih, w_hh, b_ih, b_hh, h0=None, c0=None):
    '''
    Runs a single-direction LSTM recurrence over x using raw tensor ops.

    x:  (batch, seq_len, input_size)
    w_ih: (4*hidden, input_size)  -- rows stacked as [i, f, g, o]
    w_hh: (4*hidden, hidden)      -- rows stacked as [i, f, g, o]
    b_ih, b_hh: (4*hidden,)

    Returns:
        outputs: (batch, seq_len, hidden)  -- h_t at every timestep
        (h_t, c_t): final hidden/cell state, each (batch, hidden)
    '''
    batch, seq_len, input_size = x.shape
    hidden_size = w_hh.shape[1]

    if h0 is None:
        h_t = torch.zeros(batch, hidden_size, dtype=x.dtype, device=x.device)
    else:
        h_t = h0
    if c0 is None:
        c_t = torch.zeros(batch, hidden_size, dtype=x.dtype, device=x.device)
    else:
        c_t = c0

    outputs = []

    for t in range(seq_len):
        x_t = x[:, t, :]                                   # (batch, input_size)

        # Single matmul for all 4 gates at once: (batch, 4*hidden)
        gates = x_t @ w_ih.T + b_ih + h_t @ w_hh.T + b_hh

        i_gate, f_gate, g_gate, o_gate = gates.chunk(4, dim=1)

        i_t = torch.sigmoid(i_gate)   # Input gate
        f_t = torch.sigmoid(f_gate)   # Forget gate
        g_t = torch.tanh(g_gate)      # Candidate cell state
        o_t = torch.sigmoid(o_gate)   # Output gate

        c_t = f_t * c_t + i_t * g_t   # New cell state
        h_t = o_t * torch.tanh(c_t)   # New hidden state

        outputs.append(h_t.unsqueeze(1))

    outputs = torch.cat(outputs, dim=1)   # (batch, seq_len, hidden)
    return outputs, (h_t, c_t)



## 3. Manual bidirectional LSTM layer

The backward direction runs the *same* recurrence, but on the sequence
reversed along the time axis, using its own independent set of weights. Its
per-timestep outputs are then reversed back into original time order before
concatenation, so that `output[:, t, :]` always aligns with input timestep
`t` for both directions -- matching `nn.LSTM(bidirectional=True)` semantics.

$$
y_t = [\,\overrightarrow{h}_t \;\Vert\; \overleftarrow{h}_t\,] \in \mathbb{R}^{2H}
$$


In [3]:

class ManualBiLSTM(nn.Module):
    '''
    Bidirectional LSTM built from raw tensor ops (manual_lstm_direction),
    exposing the same weight names/shapes as nn.LSTM(bidirectional=True)
    so weights can be copied 1:1 for validation.
    '''
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        def make_params(in_size):
            w_ih = nn.Parameter(torch.empty(4 * hidden_size, in_size))
            w_hh = nn.Parameter(torch.empty(4 * hidden_size, hidden_size))
            b_ih = nn.Parameter(torch.zeros(4 * hidden_size))
            b_hh = nn.Parameter(torch.zeros(4 * hidden_size))
            nn.init.xavier_uniform_(w_ih)
            nn.init.orthogonal_(w_hh)
            return w_ih, w_hh, b_ih, b_hh

        # Forward direction params
        self.w_ih_fwd, self.w_hh_fwd, self.b_ih_fwd, self.b_hh_fwd = make_params(input_size)
        # Backward direction params
        self.w_ih_bwd, self.w_hh_bwd, self.b_ih_bwd, self.b_hh_bwd = make_params(input_size)

    def forward(self, x):
        '''
        x: (batch, seq_len, input_size)
        returns: (batch, seq_len, 2*hidden_size)
        '''
        # --- Forward pass ---
        out_fwd, _ = manual_lstm_direction(
            x, self.w_ih_fwd, self.w_hh_fwd, self.b_ih_fwd, self.b_hh_fwd
        )

        # --- Backward pass: reverse time axis, run recurrence, reverse back ---
        x_rev = torch.flip(x, dims=[1])
        out_bwd_rev, _ = manual_lstm_direction(
            x_rev, self.w_ih_bwd, self.w_hh_bwd, self.b_ih_bwd, self.b_hh_bwd
        )
        out_bwd = torch.flip(out_bwd_rev, dims=[1])

        # --- Concatenate forward/backward hidden states at each timestep ---
        return torch.cat([out_fwd, out_bwd], dim=2)



## 4. Validation against `nn.LSTM(bidirectional=True)`

To prove correctness (not just plausibility), we:
1. Build a reference `nn.LSTM(bidirectional=True, batch_first=True)`.
2. Copy its weights into our `ManualBiLSTM` (matching the `[i, f, g, o]`
   stacking convention PyTorch uses internally).
3. Feed both models the same fixed-length input batch.
4. Compare output **shapes** and **values** with `torch.allclose`.


In [4]:

batch_size = 4
seq_len = 6          # fixed-length sequences
input_size = 8
hidden_size = 16

x = torch.randn(batch_size, seq_len, input_size)

# Reference PyTorch implementation
ref_lstm = nn.LSTM(
    input_size=input_size,
    hidden_size=hidden_size,
    num_layers=1,
    batch_first=True,
    bidirectional=True,
)

# Our from-scratch implementation
manual_lstm = ManualBiLSTM(input_size, hidden_size)

# --- Copy weights from ref_lstm -> manual_lstm ---
with torch.no_grad():
    # Forward direction (suffix '' in nn.LSTM param names)
    manual_lstm.w_ih_fwd.copy_(ref_lstm.weight_ih_l0)
    manual_lstm.w_hh_fwd.copy_(ref_lstm.weight_hh_l0)
    manual_lstm.b_ih_fwd.copy_(ref_lstm.bias_ih_l0)
    manual_lstm.b_hh_fwd.copy_(ref_lstm.bias_hh_l0)

    # Backward direction (suffix '_reverse' in nn.LSTM param names)
    manual_lstm.w_ih_bwd.copy_(ref_lstm.weight_ih_l0_reverse)
    manual_lstm.w_hh_bwd.copy_(ref_lstm.weight_hh_l0_reverse)
    manual_lstm.b_ih_bwd.copy_(ref_lstm.bias_ih_l0_reverse)
    manual_lstm.b_hh_bwd.copy_(ref_lstm.bias_hh_l0_reverse)

# --- Run both models ---
ref_out, (ref_h, ref_c) = ref_lstm(x)
manual_out = manual_lstm(x)

print("Reference output shape:", ref_out.shape)
print("Manual output shape:   ", manual_out.shape)
assert ref_out.shape == manual_out.shape == (batch_size, seq_len, 2 * hidden_size)

max_abs_diff = (ref_out - manual_out).abs().max().item()
print(f"Max abs difference: {max_abs_diff:.3e}")

values_match = torch.allclose(ref_out, manual_out, atol=1e-5, rtol=1e-4)
print("Values match nn.LSTM output:", values_match)
assert values_match, "Manual bidirectional LSTM does not match nn.LSTM output!"

print("\n✅ Shapes and values match nn.LSTM(bidirectional=True).")


Reference output shape: torch.Size([4, 6, 32])
Manual output shape:    torch.Size([4, 6, 32])
Max abs difference: 5.960e-08
Values match nn.LSTM output: True

✅ Shapes and values match nn.LSTM(bidirectional=True).



### Double-check: final hidden states also match

`nn.LSTM` also returns the final `(h_n, c_n)` for each direction. We can
verify these against the last timestep of our manual forward/backward
outputs (forward's last timestep == forward h_n; backward's *first*
timestep in original order == backward h_n, since the backward pass ends
at t=0).


In [5]:

# ref_h shape: (num_directions, batch, hidden) -> [0]=forward, [1]=backward
ref_h_fwd = ref_h[0]
ref_h_bwd = ref_h[1]

manual_h_fwd_last = manual_out[:, -1, :hidden_size]   # forward finishes at last timestep
manual_h_bwd_last = manual_out[:, 0, hidden_size:]    # backward finishes at first timestep

print("Forward h_n match:", torch.allclose(ref_h_fwd, manual_h_fwd_last, atol=1e-5, rtol=1e-4))
print("Backward h_n match:", torch.allclose(ref_h_bwd, manual_h_bwd_last, atol=1e-5, rtol=1e-4))


Forward h_n match: True
Backward h_n match: True



## 5. Sanity checks

A few extra checks to make sure the implementation generalizes, not just
overfits to one lucky shape/seed.


In [6]:

def check_shapes_and_values(batch_size, seq_len, input_size, hidden_size, seed):
    torch.manual_seed(seed)
    x = torch.randn(batch_size, seq_len, input_size)

    ref = nn.LSTM(input_size, hidden_size, batch_first=True, bidirectional=True)
    man = ManualBiLSTM(input_size, hidden_size)

    with torch.no_grad():
        man.w_ih_fwd.copy_(ref.weight_ih_l0); man.w_hh_fwd.copy_(ref.weight_hh_l0)
        man.b_ih_fwd.copy_(ref.bias_ih_l0);   man.b_hh_fwd.copy_(ref.bias_hh_l0)
        man.w_ih_bwd.copy_(ref.weight_ih_l0_reverse); man.w_hh_bwd.copy_(ref.weight_hh_l0_reverse)
        man.b_ih_bwd.copy_(ref.bias_ih_l0_reverse);   man.b_hh_bwd.copy_(ref.bias_hh_l0_reverse)

    ref_out, _ = ref(x)
    man_out = man(x)

    ok_shape = ref_out.shape == man_out.shape
    ok_values = torch.allclose(ref_out, man_out, atol=1e-5, rtol=1e-4)
    print(f"seed={seed:>2} batch={batch_size} seq_len={seq_len:>3} "
          f"input={input_size:>3} hidden={hidden_size:>3} -> "
          f"shape_ok={ok_shape} values_ok={ok_values}")
    return ok_shape and ok_values

configs = [
    dict(batch_size=1, seq_len=1,  input_size=5,  hidden_size=4,  seed=1),
    dict(batch_size=3, seq_len=10, input_size=12, hidden_size=7,  seed=2),
    dict(batch_size=8, seq_len=20, input_size=32, hidden_size=64, seed=3),
]

all_ok = all(check_shapes_and_values(**cfg) for cfg in configs)
print("\nAll sanity checks passed:", all_ok)
assert all_ok


seed= 1 batch=1 seq_len=  1 input=  5 hidden=  4 -> shape_ok=True values_ok=True
seed= 2 batch=3 seq_len= 10 input= 12 hidden=  7 -> shape_ok=True values_ok=True
seed= 3 batch=8 seq_len= 20 input= 32 hidden= 64 -> shape_ok=True values_ok=True

All sanity checks passed: True



### Gradient flow check

Confirm the manual layer is fully differentiable end-to-end (important since
this is meant to be usable as a real trainable layer, not just a numerical
demo).


In [7]:

torch.manual_seed(42)
model = ManualBiLSTM(input_size=10, hidden_size=6)
x = torch.randn(2, 5, 10, requires_grad=True)

out = model(x)                # (batch, seq_len, 2*hidden)
loss = out.pow(2).mean()
loss.backward()

print("Loss:", loss.item())
print("x.grad is None:", x.grad is None)
print("w_ih_fwd.grad is None:", model.w_ih_fwd.grad is None)
print("w_hh_bwd.grad is None:", model.w_hh_bwd.grad is None)

assert x.grad is not None
assert model.w_ih_fwd.grad is not None
assert model.w_hh_bwd.grad is not None
print("\n✅ Gradients flow through the manual bidirectional recurrence.")


Loss: 0.02265116572380066
x.grad is None: False
w_ih_fwd.grad is None: False
w_hh_bwd.grad is None: False

✅ Gradients flow through the manual bidirectional recurrence.



## Summary

- Implemented the four LSTM gate equations (input, forget, output, candidate
  cell state) using only raw tensor operations (`matmul`, `sigmoid`, `tanh`,
  elementwise ops).
- Built a loop-based recurrence (`manual_lstm_direction`) that processes a
  fixed-length sequence one timestep at a time, updating `(h_t, c_t)`.
- Ran this recurrence twice — once forward, once on the time-reversed
  sequence — and concatenated `[h_forward_t ; h_backward_t]` at every
  timestep to build a `ManualBiLSTM` layer.
- Validated the implementation against PyTorch's built-in
  `nn.LSTM(bidirectional=True)` by copying weights in matching `[i, f, g, o]`
  order and comparing output shapes, output values, final hidden states, and
  gradient flow — all matched within floating-point tolerance.
